# 🔥 Forest Fire Detection Using CNN

This notebook uses a trained CNN model to detect fire in a live webcam feed. When fire is detected above the configured confidence threshold, it displays an alert and can optionally send an email notification.

**Before running:** Place `fire_detection_model.h5` in the notebook's working directory. The model is expected to accept images resized to **224 × 224** and return a binary prediction score.

## 1. Install dependencies
Run this cell once if the required packages are not installed.

In [ ]:
%pip install opencv-python numpy tensorflow

## 2. Import libraries

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Optional: email notifications use Python's built-in email and smtplib modules.
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Windows-only sound alert. On other operating systems, sound alerts are skipped.
try:
    import winsound
except ImportError:
    winsound = None


## 3. Load the trained CNN model

In [ ]:
MODEL_PATH = "fire_detection_model.h5"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model file not found: {MODEL_PATH}. "
        "Place the trained model in the notebook's working directory."
    )

model = load_model(MODEL_PATH)
print("CNN model loaded successfully.")

## 4. Configure email alerts (optional)

For Gmail, use an **App Password** rather than your normal account password. Keep credentials private; do not upload them to GitHub. You can leave `ENABLE_EMAIL_ALERTS` as `False` to run detection without email.

In [ ]:
ENABLE_EMAIL_ALERTS = False

# Set these values locally if you enable email alerts.
EMAIL_SENDER = os.getenv("FIRE_ALERT_SENDER", "")
EMAIL_PASSWORD = os.getenv("FIRE_ALERT_APP_PASSWORD", "")
EMAIL_RECEIVER = os.getenv("FIRE_ALERT_RECEIVER", "")

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

def send_email_alert():
    """Send an email notification when fire is detected."""
    if not ENABLE_EMAIL_ALERTS:
        print("Email alerts are disabled.")
        return

    if not all([EMAIL_SENDER, EMAIL_PASSWORD, EMAIL_RECEIVER]):
        print("Email alert skipped: configure sender, app password, and receiver.")
        return

    message = MIMEMultipart()
    message["From"] = EMAIL_SENDER
    message["To"] = EMAIL_RECEIVER
    message["Subject"] = "🔥 Forest Fire Alert"
    message.attach(MIMEText(
        "Fire has been detected in the monitored area. Please investigate.",
        "plain"
    ))

    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.sendmail(EMAIL_SENDER, EMAIL_RECEIVER, message.as_string())
        print("Email alert sent.")
    except Exception as error:
        print("Error sending email alert:", error)


## 5. Prepare images for prediction

In [ ]:
def preprocess_frame(frame):
    """Resize and normalize a webcam frame for the CNN model."""
    image = cv2.resize(frame, (224, 224))
    image = image.astype("float32") / 255.0
    image = np.expand_dims(image, axis=0)
    return image


## 6. Run live fire detection

Run this cell to open the default webcam. Press **`e`** in the video window to stop detection. If your model uses a different input size, preprocessing, or output convention, update this section to match its training setup.

In [ ]:
CLASS_NAMES = ["No Fire", "Fire"]
CONFIDENCE_THRESHOLD = 50.0

# Change to 1 or another camera index if your default camera is not camera 0.
CAMERA_INDEX = 0

cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Check camera permissions or CAMERA_INDEX.")

fire_detected = False

try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Could not read a frame from the webcam.")
            break

        processed_frame = preprocess_frame(frame)
        prediction = model.predict(processed_frame, verbose=0)

        # This assumes the model returns one sigmoid score:
        # values above 0.5 indicate Fire; values at or below 0.5 indicate No Fire.
        fire_score = float(np.asarray(prediction).reshape(-1)[0])
        if fire_score > 0.5:
            label = "Fire"
            confidence = fire_score * 100
        else:
            label = "No Fire"
            confidence = (1 - fire_score) * 100

        display_text = f"{label}: {confidence:.2f}%"
        display_color = (0, 0, 255) if label == "Fire" else (0, 255, 0)

        cv2.putText(
            frame, display_text, (10, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1, display_color, 2
        )
        cv2.imshow("Forest Fire Detection", frame)

        if label == "Fire" and confidence >= CONFIDENCE_THRESHOLD:
            if winsound is not None:
                winsound.Beep(1000, 500)

            if not fire_detected:
                send_email_alert()
                fire_detected = True
        else:
            fire_detected = False

        # Press 'e' in the OpenCV window to exit.
        if cv2.waitKey(1) & 0xFF == ord("e"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam resources released.")


## 7. Notes

- This notebook performs live webcam inference; it does not train the CNN model.
- The trained model file (`fire_detection_model.h5`) is required.
- The code assumes a single sigmoid output where a score above `0.5` represents **Fire**. Confirm this matches your model's training labels.
- `winsound` works only on Windows. On other systems, visual detection still works, but the sound alert is skipped.
- Test the system in a controlled environment. Image-based predictions can produce false positives or miss real fires.